In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = Path(r"C:\desktop\agent_contest")

load_dotenv(PROJECT_ROOT / ".env", override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(".env에 OPENAI_API_KEY를 설정해주세요.")

client = OpenAI()
print("API 키 로딩 및 클라이언트 준비 완료")

API 키 로딩 및 클라이언트 준비 완료


In [2]:
models = sorted(m.id for m in client.models.list())
gpt_models = [m for m in models if m.startswith("gpt")]

print(f"전체 {len(models)}개 중 gpt 계열 {len(gpt_models)}개\n")
for m in gpt_models:
    print(m)

전체 96개 중 gpt 계열 82개

gpt-4.1
gpt-4.1-2025-04-14
gpt-4.1-mini
gpt-4.1-mini-2025-04-14
gpt-4o
gpt-4o-2024-08-06
gpt-4o-2024-11-20
gpt-4o-mini
gpt-4o-mini-2024-07-18
gpt-4o-mini-transcribe
gpt-4o-mini-transcribe-2025-03-20
gpt-4o-mini-transcribe-2025-12-15
gpt-4o-mini-tts
gpt-4o-mini-tts-2025-03-20
gpt-4o-mini-tts-2025-12-15
gpt-4o-transcribe
gpt-4o-transcribe-diarize
gpt-5
gpt-5-2025-08-07
gpt-5-mini
gpt-5-mini-2025-08-07
gpt-5-nano
gpt-5-nano-2025-08-07
gpt-5-pro
gpt-5-pro-2025-10-06
gpt-5-search-api
gpt-5-search-api-2025-10-14
gpt-5.1
gpt-5.1-2025-11-13
gpt-5.2
gpt-5.2-2025-12-11
gpt-5.2-pro
gpt-5.2-pro-2025-12-11
gpt-5.3-codex
gpt-5.4
gpt-5.4-2026-03-05
gpt-5.4-mini
gpt-5.4-mini-2026-03-17
gpt-5.4-nano
gpt-5.4-nano-2026-03-17
gpt-5.4-pro
gpt-5.4-pro-2026-03-05
gpt-5.5
gpt-5.5-2026-04-23
gpt-5.5-pro
gpt-5.5-pro-2026-04-23
gpt-5.6-luna
gpt-5.6-sol
gpt-5.6-terra
gpt-6-astra
gpt-6-luna
gpt-6-sol
gpt-6.1-sol
gpt-6.1-terra
gpt-audio
gpt-audio-1.5
gpt-audio-2025-08-28
gpt-audio-mini
gpt-audi

In [3]:
MODEL = "gpt-4o-mini"
print("목록에 있음:", MODEL in models)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "한 문장으로 답해: ESG의 G는 무엇의 약자야?"}],
    temperature=0,
)
print(resp.choices[0].message.content)
print("사용 토큰:", resp.usage.total_tokens)

목록에 있음: True
ESG의 G는 "Governance"의 약자입니다.
사용 토큰: 38


## baseline 데이터 불러오기
02에서 만든 안건표, 파일 목록표(manifest), 원문 조각(chunks)을 불러온다.

In [4]:
import pandas as pd

PROC = PROJECT_ROOT / "data/processed"

agenda   = pd.read_csv(PROC / "googl_agenda_2026.csv")
manifest = pd.read_csv(PROC / "googl_2026_manifest.csv")
chunks   = pd.read_csv(PROC / "googl_2026_chunks.csv")

print("안건:", agenda.shape, "/ manifest:", manifest.shape, "/ 조각:", chunks.shape)

안건: (14, 9) / manifest: (52, 5) / 조각: (331, 6)


## 안건별 LLM 입력 만들기
manifest에서 llm_input=True인 파일의 조각을 모아 "[조각ID] 본문" 형태로 이어 붙인다.

In [5]:
ROLE_ORDER = ["main", "holder", "company", "overview"]   # 나머지(supp_*)는 그 뒤

def build_context(no):
    """안건 번호 → LLM에 넣을 원문 (조각 ID 포함)"""
    m = manifest[(manifest["proposal_no"] == no) & manifest["llm_input"]].copy()
    m["rank"] = m["role"].apply(lambda r: ROLE_ORDER.index(r) if r in ROLE_ORDER else len(ROLE_ORDER))
    files = m.sort_values(["rank", "file"])["file"].tolist()

    sel = chunks[chunks["file"].isin(files)].copy()
    sel["file_order"] = sel["file"].map({f: i for i, f in enumerate(files)})
    sel = sel.sort_values(["file_order", "start"])
    return "\n\n".join(f"[{r.chunk_id}] {r.text}" for r in sel.itertuples()), len(sel)

ctx5, n5 = build_context(5)
print(f"5번 입력: 조각 {n5}개 / {len(ctx5):,}자\n")
print(ctx5[:300])

5번 입력: 조각 28개 / 16,126자

[p05_holder_001] Trillium ESG Global Equity Fund, as lead filer, along with other co-filers, whose names, addresses, and shareholdings will be provided by us upon request, have advised us that they intend to submit the following proposal for consideration at our Annual Meeting. Enhanced Disclosure W


## 지시문(프롬프트)과 결과 양식
LLM이 지켜야 할 규칙과 JSON 결과 양식을 정한다. 찬반 의견은 참고용이며 최종 결정은 담당자가 한다.

In [19]:
SYSTEM_PROMPT = """너는 기관투자자의 의결권 검토를 돕는 ESG 분석 보조자다.
주어진 위임장 원문 조각만 근거로 안건을 분석한다.

규칙
1. 원문 조각에 없는 내용은 쓰지 않는다. 외부 지식으로 사실을 추가하지 않는다.
2. 모든 요약과 쟁점에는 근거를 단다. 근거는 조각 ID와 인용문이다.
   인용문은 그 조각에서 연속된 한 구간을 글자 그대로 복사한다(20단어 이내).
   "..."로 생략하거나 여러 구간을 이어 붙이지 않는다.
3. 주주의 주장(holder 조각)과 회사의 주장(company 조각)을 섞지 않는다.
4. 근거를 찾지 못한 판단은 needs_check에 적는다.
5. 찬반 의견(vote_opinion)은 장기 주주가치와 ESG 위험 관점의 참고 의견이다.
   회사 주장을 그대로 따르지 말고, 주주가 제기한 쟁점에 회사가 실제로 답했는지 따져서 판단한다.
   이유에는 주주 쪽 근거와 회사 쪽 근거를 모두 단다. 근거가 부족하면 "판단보류"로 한다.
6. 요약과 이유는 한국어로, 인용문은 원문 영어 그대로 쓴다.
7. key_issues는 주주와 회사의 주장이 부딪히는 지점이다. 쟁점마다 holder 근거와 company 근거를 하나 이상씩 단다.
8. 회사가 "이미 공시하고 있다"고 주장하지만 위임장 안에서 그 내용을 확인할 수 없으면 needs_check에 적는다.
9. 회사가 제안한 안건(제안 주체: 회사)이면 holder_request에는 "안건 내용"을 요약하고 main·supp 조각을 근거로 단다.
   company_response에는 이사회가 찬성을 권하는 이유를 요약한다.
   key_issues는 안건 내용과 보조 자료(이사 독립성, 감사 보수, 보수 결정 등)에서 판단이 필요한 지점을 적는다.

E/S/G 분류 기준 (위에서부터 순서대로 적용)
- 회사 안건(이사 선임, 감사인, 경영진 보수, 주식보상, 정관)은 G
- 주주제안은 "무엇을 바꾸라고 요구하는가"로 1차 분류를 정한다.
  이사회 구조, 위원회, 의결권, 보수를 바꾸라는 요구 → G
  어떤 영향에 대한 보고서나 정책을 요구 → 그 영향이 환경(기후, 물, 에너지)이면 E,
  사람과 사회(개인정보, 인권, 노동·인력, 콘텐츠·표현, 정치 활동)이면 S
- 다른 영역에도 걸치면 esg_secondary에 적고, 없으면 "없음"
- 제안자의 성향은 분류에 반영하지 않는다.
- esg_reason에 분류 이유를 한 문장으로 쓴다.
"""

def build_user_message(no):
    row = agenda.loc[agenda["proposal_no"] == no].iloc[0]
    ctx, _ = build_context(no)
    header = (f"안건 {no}번: {row['title']}\n"
              f"제안 주체: {row['proposer']} / 제안자: {row.get('filer', '')}\n"
              f"이사회 권고: {row['board_rec']}\n\n"
              f"=== 원문 조각 ===\n")
    return header + ctx

print(build_user_message(5)[:300])

안건 5번: Shareholder Proposal Regarding an Enhanced Disclosure on Climate Goals
제안 주체: 주주 / 제안자: Trillium ESG Global Equity Fund
이사회 권고: AGAINST

=== 원문 조각 ===
[p05_holder_001] Trillium ESG Global Equity Fund, as lead filer, along with other co-filers, whose names, addresses, and shareholdings will be


## 5번 안건 LLM 분석 (시험)
5번 안건 하나를 gpt-4o-mini에 보내 JSON 검토표를 받고, 파일로 저장한다.

In [7]:
import json

def review_proposal(no):
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON으로만 답하게 강제
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(no)},
        ],
    )
    result = json.loads(resp.choices[0].message.content)
    result["proposal_no"] = no
    return result, resp.usage

r5, usage5 = review_proposal(5)

OUT_DIR = PROC / "llm_review"
OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / "googl_p05.json").write_text(json.dumps(r5, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"토큰: 입력 {usage5.prompt_tokens:,} / 출력 {usage5.completion_tokens:,}\n")
print(json.dumps(r5, ensure_ascii=False, indent=2))

토큰: 입력 3,708 / 출력 636

{
  "esg_primary": "E",
  "esg_secondary": "G",
  "holder_request": {
    "summary": "주주는 Alphabet이 기후 변화 관련 목표를 달성하기 위한 계획과 진행 상황에 대한 투명성을 높일 것을 요구하고 있다.",
    "evidence": [
      {
        "chunk_id": "p05_holder_006",
        "quote": "Shareholders request that Alphabet publish a report, at reasonable cost, within a reasonable time, and excluding confidential or proprietary information, explaining how it will meet the climate change-related commitments it has made on GHGs."
      }
    ]
  },
  "company_response": {
    "summary": "회사는 이미 기후 계획과 진행 상황에 대한 상세한 정보를 제공하고 있으며, 추가 보고서는 불필요하다고 주장하고 있다.",
    "evidence": [
      {
        "chunk_id": "p05_company_001",
        "quote": "Our existing environmental reporting has long provided detailed and extensive information about our climate plans, progress, policies, and practices."
      }
    ]
  },
  "key_issues": [
    {
      "issue": "기후 목표 달성을 위한 투명성 요구",
      "evidence": [
        {
          "chunk_id": "

## 결과 확인 (쟁점, 찬반 의견)
출력이 길어 잘리는 부분(key_issues, vote_opinion, needs_check)만 따로 본다.

In [8]:
for k in ["key_issues", "proponent_note", "vote_opinion", "needs_check"]:
    print(f"===== {k} =====")
    print(json.dumps(r5.get(k), ensure_ascii=False, indent=2), "\n")

===== key_issues =====
[
  {
    "issue": "기후 목표 달성을 위한 투명성 요구",
    "evidence": [
      {
        "chunk_id": "p05_holder_004",
        "quote": "investors remain in the dark about the full scope, scale, and timelines of these projects and thus, the likelihood of Alphabet meeting its 2030 goals."
      },
      {
        "chunk_id": "p05_company_005",
        "quote": "Preparing the separate, bespoke report as requested by the proponent would be duplicative and inefficient, administratively burdensome, and an ineffective use of our resources."
      }
    ]
  }
] 

===== proponent_note =====
"제안자는 Trillium ESG Global Equity Fund로, 기후 변화 관련 목표에 대한 투명성을 높이기 위한 제안을 하고 있다." 

===== vote_opinion =====
{
  "opinion": "찬성",
  "reason": "주주는 Alphabet의 기후 목표 달성을 위한 구체적인 계획과 진행 상황에 대한 투명성을 요구하고 있으며, 이는 장기적인 주주 가치를 높이는 데 기여할 수 있다. 회사는 이미 정보를 제공하고 있다고 주장하지만, 주주가 제기한 투명성 부족 문제에 대한 구체적인 해결책을 제시하지 않았다.",
  "evidence": [
    {
      "chunk_id": "p05_holder_003",
      "quote": "Alphabet identifies th

## 근거 검증
LLM이 단 근거마다 (1) 조각 ID가 있는지, (2) 인용문이 그 조각에 그대로 있는지, (3) 주주/회사 구분이 맞는지 확인한다.

In [11]:
import re

chunk_text = dict(zip(chunks["chunk_id"], chunks["text"]))

def norm(s):
    """비교용: 따옴표 제거, 공백 정리, 소문자, 끝 문장부호 제거"""
    s = re.sub(r"[\"'“”‘’]", "", s)          # 모든 종류의 따옴표 제거
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s.rstrip(".,;:")

def verify(result):
    rows = []
    sections = [("holder_request", result.get("holder_request", {}).get("evidence", [])),
                ("company_response", result.get("company_response", {}).get("evidence", [])),
                ("vote_opinion", result.get("vote_opinion", {}).get("evidence", []))]
    for i, ki in enumerate(result.get("key_issues", []), 1):
        sections.append((f"key_issue_{i}", ki.get("evidence", [])))

    for sec, evs in sections:
        for ev in evs:
            cid, q = ev.get("chunk_id", ""), ev.get("quote", "")
            exists = cid in chunk_text
            quote_ok = exists and "..." not in q and "…" not in q and norm(q) in norm(chunk_text[cid])
            if sec == "holder_request":
                role_ok = "_holder_" in cid
            elif sec == "company_response":
                role_ok = "_company_" in cid
            else:
                role_ok = True
            rows.append({"section": sec, "chunk_id": cid, "id_ok": exists,
                         "quote_ok": quote_ok, "role_ok": role_ok, "quote": q})
    return pd.DataFrame(rows)

v5 = verify(r5)
print(f"근거 {len(v5)}개 중 모두 통과: {(v5[['id_ok','quote_ok','role_ok']].all(axis=1)).sum()}개\n")
v5

근거 6개 중 모두 통과: 6개



,section,chunk_id,id_ok,quote_ok,role_ok,quote
0,holder_request,p05_holder_006,True,True,True,Shareholders request that Alphabet publish a r...
1,company_response,p05_company_001,True,True,True,Our existing environmental reporting has long ...
2,vote_opinion,p05_holder_003,True,True,True,Alphabet identifies the growing demand for its...
3,vote_opinion,p05_company_003,True,True,True,We also shared in our annual Environmental Rep...
4,key_issue_1,p05_holder_004,True,True,True,investors remain in the dark about the full sc...
5,key_issue_1,p05_company_005,True,True,True,"Preparing the separate, bespoke report as requ..."


## 검증 실패 원인 확인
인용문이 원문과 어디서 달라졌는지, 원문에서 가장 비슷한 구간과 나란히 본다.

In [10]:
from difflib import SequenceMatcher

print("===== 찬반 의견 =====")
print(r5["vote_opinion"]["opinion"], "/", r5["vote_opinion"]["reason"], "\n")

def closest_span(quote, text):
    """원문에서 인용문과 가장 비슷한 구간 찾기"""
    q, t = norm(quote), norm(text)
    m = SequenceMatcher(None, t, q).find_longest_match(0, len(t), 0, len(q))
    start = max(0, m.a - m.b)
    return text[start:start + len(quote) + 20], round(m.size / max(len(q), 1), 2)

for r in v5[~v5["quote_ok"]].itertuples():
    full_q = next(ev["quote"] for sec, evs in [
        ("holder_request", r5["holder_request"]["evidence"]),
        ("company_response", r5["company_response"]["evidence"]),
        ("vote_opinion", r5["vote_opinion"]["evidence"])] + [
        (f"key_issue_{i}", k["evidence"]) for i, k in enumerate(r5["key_issues"], 1)]
        for ev in evs if sec == r.section and ev["chunk_id"] == r.chunk_id)
    span, ratio = closest_span(full_q, chunk_text[r.chunk_id])
    print(f"===== {r.section} / {r.chunk_id} (일치 비율 {ratio}) =====")
    print("LLM 인용 :", full_q)
    print("원문 구간:", span, "\n")

===== 찬반 의견 =====
찬성 / 주주는 Alphabet의 기후 목표 달성을 위한 구체적인 계획과 진행 상황에 대한 투명성을 요구하고 있으며, 이는 장기적인 주주 가치를 높이는 데 기여할 수 있다. 회사는 이미 정보를 제공하고 있다고 주장하지만, 주주가 제기한 투명성 부족 문제에 대한 구체적인 해결책을 제시하지 않았다. 

===== vote_opinion / p05_holder_003 (일치 비율 0.25) =====
LLM 인용 : Alphabet identifies the growing demand for its digital services, including AI, as 'putting pressure on the amount of power that [it] need[s],' and acknowledges facing challenges and complexities in meeting its climate commitments.
원문 구간: Alphabet identifies the growing demand for its digital services, including AI, as “putting pressure on the amount of power that [it] need[s],” and acknowledges facing challenges and complexities in meeting its climate commitments. 6 Although the comp 



## 검증 결과 반영
근거 검증에 실패한 항목을 needs_check에 "[근거 확인 필요]"로 자동 추가하고, 통과 개수를 결과에 기록한다.

In [12]:
def apply_verification(result, v):
    """검증 실패 근거를 needs_check에 추가하고, 통과 개수를 기록"""
    result = dict(result)
    ok = v[["id_ok", "quote_ok", "role_ok"]].all(axis=1)

    needs = list(result.get("needs_check", []))
    for r in v[~ok].itertuples():
        reasons = [name for name, passed in [("조각 ID 없음", r.id_ok),
                                              ("인용문 불일치", r.quote_ok),
                                              ("주주/회사 구분 오류", r.role_ok)] if not passed]
        needs.append(f"[근거 확인 필요] {r.section} / {r.chunk_id}: {', '.join(reasons)}")

    result["needs_check"] = needs
    result["evidence_total"] = int(len(v))
    result["evidence_passed"] = int(ok.sum())
    return result

# 1) 실제 5번 결과에 적용 (모두 통과라 needs_check는 그대로)
r5_checked = apply_verification(r5, v5)
print("통과:", r5_checked["evidence_passed"], "/", r5_checked["evidence_total"])
print("needs_check:", r5_checked["needs_check"], "\n")

# 2) 동작 확인용: 인용문 하나를 일부러 틀리게 바꿔서 시험
import copy
fake = copy.deepcopy(r5)
fake["company_response"]["evidence"][0]["quote"] = "This sentence does not exist in the proxy."
fake_checked = apply_verification(fake, verify(fake))
print("[시험] 통과:", fake_checked["evidence_passed"], "/", fake_checked["evidence_total"])
print("[시험] needs_check:", fake_checked["needs_check"])

# 3) 검증 반영본 저장
(OUT_DIR / "googl_p05.json").write_text(json.dumps(r5_checked, ensure_ascii=False, indent=2), encoding="utf-8")

통과: 6 / 6
needs_check: [] 

[시험] 통과: 5 / 6
[시험] needs_check: ['[근거 확인 필요] company_response / p05_company_001: 인용문 불일치']


2452

## 14개 안건 전체 실행
모든 안건을 LLM에 보내고, 근거 검증 결과를 반영해 안건별 JSON으로 저장한다. 회사 안건은 주주/회사 구분 기준을 따로 적용한다.

In [13]:
import time

def verify_any(result):
    """주주제안은 verify 그대로, 회사 안건은 주주/회사 구분 기준만 바꿔서 검증"""
    v = verify(result)
    proposer = agenda.loc[agenda["proposal_no"] == result["proposal_no"], "proposer"].iloc[0]
    if proposer == "회사":
        is_hr = v["section"] == "holder_request"
        v.loc[is_hr, "role_ok"] = v.loc[is_hr, "chunk_id"].str.contains("_main_|_supp_")
        v.loc[v["section"] == "company_response", "role_ok"] = True
    return v

all_results, summary_rows = {}, []
for no in agenda["proposal_no"]:
    try:
        r, usage = review_proposal(no)
        r = apply_verification(r, verify_any(r))
        all_results[no] = r
        (OUT_DIR / f"googl_p{no:02d}.json").write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")
        summary_rows.append({
            "no": no,
            "esg": f"{r.get('esg_primary')}/{r.get('esg_secondary')}",
            "opinion": r.get("vote_opinion", {}).get("opinion"),
            "board_rec": agenda.loc[agenda["proposal_no"] == no, "board_rec"].iloc[0],
            "evidence": f"{r['evidence_passed']}/{r['evidence_total']}",
            "needs_check": len(r["needs_check"]),
            "tokens": usage.total_tokens,
        })
        print(f"{no}번 완료")
    except Exception as e:
        print(f"{no}번 실패: {e}")
    time.sleep(1)

summary = pd.DataFrame(summary_rows)
print("\n총 토큰:", summary["tokens"].sum())
summary

1번 완료
2번 완료
3번 완료
4번 완료
5번 완료
6번 완료
7번 완료
8번 완료
9번 완료
10번 완료
11번 완료
12번 완료
13번 완료
14번 완료

총 토큰: 68470


,no,esg,opinion,board_rec,evidence,needs_check,tokens
0,1,G/없음,찬성,FOR,2/5,3,9063
1,2,없음/없음,찬성,FOR,3/3,0,2392
2,3,없음/없음,찬성,FOR,4/5,1,5500
3,4,S/G,반대,FOR,7/7,0,7407
4,5,E/G,반대,AGAINST,4/5,1,4207
5,6,E/S | G,반대,AGAINST,0/6,6,4091
6,7,G/없음,반대,AGAINST,6/6,0,4261
7,8,G/없음,반대,AGAINST,7/7,0,4367
8,9,G/없음,찬성,AGAINST,2/6,4,4245
9,10,S/없음,반대,AGAINST,4/5,1,4031


## 13. 전체 검증 실패 원인 집계
14개 결과에서 실패한 근거를 모아, 실패 이유별 개수와 6번 안건의 실패 내용을 본다.

In [14]:
fail_rows = []
for no, r in all_results.items():
    v = verify_any(r)
    for row in v[~v[["id_ok", "quote_ok", "role_ok"]].all(axis=1)].itertuples():
        reason = ("조각 ID 없음" if not row.id_ok else
                  "인용문 불일치" if not row.quote_ok else "주주/회사 구분 오류")
        fail_rows.append({"no": no, "section": row.section, "chunk_id": row.chunk_id,
                          "reason": reason, "quote": row.quote[:70]})

fails = pd.DataFrame(fail_rows)
print("실패 이유별 개수")
print(fails["reason"].value_counts(), "\n")
fails[fails["no"] == 6]

실패 이유별 개수
reason
인용문 불일치    25
Name: count, dtype: int64 



,no,section,chunk_id,reason,quote
5,6,holder_request,p06_holder_006,인용문 불일치,Resolved: Shareholders request that Alphabet I...
6,6,company_response,p06_company_001,인용문 불일치,Our Board of Directors recommends a vote AGAIN...
7,6,vote_opinion,p06_holder_004,인용문 불일치,By focusing on operational resilience and stra...
8,6,vote_opinion,p06_company_002,인용문 불일치,Our existing disclosures provide details of ou...
9,6,key_issue_1,p06_holder_002,인용문 불일치,Water scarcity poses a material risk to Alphab...
10,6,key_issue_1,p06_company_007,인용문 불일치,"Preparing a separate, bespoke report as reques..."


## 14. 인용문 불일치 유형 분류
실패한 인용문이 사소한 차이인지, 다른 조각에 있는지(ID 오류), 바꿔 쓴 것인지 나눈다.

In [15]:
def match_ratio(quote, text):
    """인용문 중 원문과 연속으로 같은 부분의 비율"""
    q, t = norm(quote), norm(text)
    m = SequenceMatcher(None, t, q).find_longest_match(0, len(t), 0, len(q))
    return m.size / max(len(q), 1)

def full_quote(r, sec, cid):
    """결과 JSON에서 해당 근거의 전체 인용문 꺼내기"""
    if sec.startswith("key_issue_"):
        evs = r["key_issues"][int(sec.split("_")[-1]) - 1]["evidence"]
    else:
        evs = r[sec]["evidence"]
    return next(ev["quote"] for ev in evs if ev["chunk_id"] == cid)

types = []
for f in fails.itertuples():
    q = full_quote(all_results[f.no], f.section, f.chunk_id)
    own = match_ratio(q, chunk_text[f.chunk_id])

    # 같은 안건의 다른 조각에서 찾기
    prefix = f"p{f.no:02d}_"
    found_in = [cid for cid, t in chunk_text.items()
                if cid.startswith(prefix) and cid != f.chunk_id and norm(q) in norm(t)]

    if found_in:
        kind = "(나) ID만 틀림"
    elif own >= 0.9:
        kind = "(가) 사소한 차이"
    else:
        kind = "(다) 바꿔 씀"
    types.append({"no": f.no, "section": f.section, "chunk_id": f.chunk_id,
                  "own_ratio": round(own, 2), "found_in": ", ".join(found_in), "kind": kind})

types = pd.DataFrame(types)
print(types["kind"].value_counts(), "\n")
types

kind
(가) 사소한 차이    17
(다) 바꿔 씀       5
(나) ID만 틀림     3
Name: count, dtype: int64 



,no,section,chunk_id,own_ratio,found_in,kind
0,1,company_response,p01_supp_independence_001,0.26,p01_supp_leadership_003,(나) ID만 틀림
1,1,vote_opinion,p01_supp_independence_001,0.26,p01_supp_leadership_003,(나) ID만 틀림
2,1,key_issue_1,p01_supp_independence_001,1.00,,(가) 사소한 차이
3,3,company_response,p03_main_034,0.34,p03_main_035,(나) ID만 틀림
4,5,key_issue_1,p05_holder_006,0.62,,(다) 바꿔 씀
5,6,holder_request,p06_holder_006,1.00,,(가) 사소한 차이
6,6,company_response,p06_company_001,1.00,,(가) 사소한 차이
7,6,vote_opinion,p06_holder_004,1.00,,(가) 사소한 차이
8,6,vote_opinion,p06_company_002,1.00,,(가) 사소한 차이
9,6,key_issue_1,p06_holder_002,1.00,,(가) 사소한 차이


## 15. 검증 기준 보정 및 재집계
95% 이상 일치는 통과, 다른 조각에 있는 인용은 ID를 자동 교정한다. 나머지만 확인 필요로 남긴다.

In [18]:
def clean_quote(q):
    """끝에 붙은 ... 또는 … 제거 (문장이 이어진다는 표시라 허용)"""
    return re.sub(r"\s*(\.\.\.|…)\s*$", "", q).strip()

def fix_and_verify(result, threshold=0.95):
    """근거마다: 통과 / ID 교정 / 실패. 끝의 ...는 허용, 중간의 ...는 실패"""
    result = copy.deepcopy(result)
    no = result["proposal_no"]
    prefix = f"p{no:02d}_"
    rows = []

    groups = [("holder_request", result.get("holder_request", {}).get("evidence", [])),
              ("company_response", result.get("company_response", {}).get("evidence", [])),
              ("vote_opinion", result.get("vote_opinion", {}).get("evidence", []))]
    groups += [(f"key_issue_{i}", k.get("evidence", [])) for i, k in enumerate(result.get("key_issues", []), 1)]

    for sec, evs in groups:
        for ev in evs:
            cid = ev.get("chunk_id_original", ev.get("chunk_id", ""))   # 이전 교정 전 ID부터 다시 검사
            q = clean_quote(ev.get("quote", ""))
            status = "실패"
            if cid in chunk_text and "..." not in q and "…" not in q:
                if match_ratio(q, chunk_text[cid]) >= threshold:
                    status = "통과"
                    ev["chunk_id"] = cid
                    ev.pop("chunk_id_original", None)
                else:
                    other = next((c for c, t in chunk_text.items()
                                  if c.startswith(prefix) and match_ratio(q, t) >= threshold), None)
                    if other:
                        ev["chunk_id_original"] = cid
                        ev["chunk_id"] = other
                        status = "ID 교정"
            ev["check"] = status
            rows.append({"section": sec, "chunk_id": ev["chunk_id"], "status": status})

    v = pd.DataFrame(rows)
    needs = [n for n in result.get("needs_check", []) if not n.startswith("[근거 확인 필요]")]
    needs += [f"[근거 확인 필요] {r.section} / {r.chunk_id}: 인용문이 원문과 다름"
              for r in v[v["status"] == "실패"].itertuples()]
    result["needs_check"] = needs
    result["evidence_total"] = len(v)
    result["evidence_passed"] = int((v["status"] != "실패").sum())
    return result, v

rows = []
for no, r in all_results.items():
    r2, v2 = fix_and_verify(r)
    all_results[no] = r2
    (OUT_DIR / f"googl_p{no:02d}.json").write_text(json.dumps(r2, ensure_ascii=False, indent=2), encoding="utf-8")
    rows.append({"no": no, **v2["status"].value_counts().to_dict()})

recount = pd.DataFrame(rows).fillna(0).set_index("no").astype(int)
print("전체:", recount.sum().to_dict())
recount

전체: {'ID 교정': 4, '통과': 71, '실패': 7}


,ID 교정,통과,실패
no,,,
1,3,2,0
2,0,3,0
3,1,4,0
4,0,7,0
5,0,3,2
6,0,6,0
7,0,6,0
8,0,7,0
9,0,6,0


## 16. 실패 근거 직접 확인
실패로 남은 근거의 인용문 전체와 "..." 포함 여부, 원래 조각과의 일치 비율을 본다.

In [17]:
check_rows = []
for no, r in all_results.items():
    groups = [("holder_request", r.get("holder_request", {}).get("evidence", [])),
              ("company_response", r.get("company_response", {}).get("evidence", [])),
              ("vote_opinion", r.get("vote_opinion", {}).get("evidence", []))]
    groups += [(f"key_issue_{i}", k.get("evidence", [])) for i, k in enumerate(r.get("key_issues", []), 1)]
    for sec, evs in groups:
        for ev in evs:
            if ev.get("check") == "실패":
                q, cid = ev["quote"], ev["chunk_id"]
                check_rows.append({
                    "no": no, "section": sec, "chunk_id": cid,
                    "ellipsis": ("..." in q) or ("…" in q),
                    "ratio": round(match_ratio(q, chunk_text.get(cid, "")), 2),
                    "quote": q,
                })

failed = pd.DataFrame(check_rows)
print("실패:", len(failed), "/ '...' 포함:", failed["ellipsis"].sum(), "\n")
pd.set_option("display.max_colwidth", 200)
failed[failed["no"].isin([6, 11])]

실패: 23 / '...' 포함: 18 



,no,section,chunk_id,ellipsis,ratio,quote
2,6,holder_request,p06_holder_006,True,1.0,"Resolved: Shareholders request that Alphabet Inc. prepare and publicly disclose a report, or disclose analysis which has already been performed..."
3,6,company_response,p06_company_001,True,1.0,Our Board of Directors recommends a vote AGAINST the shareholder proposal because: • We have a long track record of providing extensive information...
4,6,vote_opinion,p06_holder_004,True,1.0,"By focusing on operational resilience and strategic resource allocation, Alphabet can protect shareholder interests..."
5,6,vote_opinion,p06_company_002,True,1.0,"Our existing disclosures provide details of our water strategy, and we provide extensive reporting through our annual Environmental Report..."
6,6,key_issue_1,p06_holder_002,True,1.0,Water scarcity poses a material risk to Alphabet’s infrastructure and brand reputation...
7,6,key_issue_1,p06_company_007,True,1.0,"Preparing a separate, bespoke report as requested by the proponent would be duplicative and inefficient..."
13,11,holder_request,p11_holder_001,False,0.0,"Shareholders request the Board of Directors issue public reporting, prepared at reasonable cost and omitting proprietary information, assessing operational, reputational, regulatory and legal risk..."


## 17. 구조화 출력으로 재실행
JSON 스키마로 E/S/G 값을 "E, S, G" 중 하나로 강제하고 14개 안건을 다시 실행한다. 결과는 llm_review_v2에 따로 저장해 이전 결과와 비교한다.

In [20]:
EVID = {"type": "array", "items": {
    "type": "object",
    "properties": {"chunk_id": {"type": "string"}, "quote": {"type": "string"}},
    "required": ["chunk_id", "quote"], "additionalProperties": False}}

def obj(props):
    return {"type": "object", "properties": props,
            "required": list(props), "additionalProperties": False}

REVIEW_SCHEMA = obj({
    "esg_primary":   {"type": "string", "enum": ["E", "S", "G"]},
    "esg_secondary": {"type": "string", "enum": ["E", "S", "G", "없음"]},
    "esg_reason":    {"type": "string"},
    "holder_request":   obj({"summary": {"type": "string"}, "evidence": EVID}),
    "company_response": obj({"summary": {"type": "string"}, "evidence": EVID}),
    "key_issues": {"type": "array", "items": obj({"issue": {"type": "string"}, "evidence": EVID})},
    "proponent_note": {"type": "string"},
    "vote_opinion": obj({"opinion": {"type": "string", "enum": ["찬성", "반대", "판단보류"]},
                         "reason": {"type": "string"}, "evidence": EVID}),
    "needs_check": {"type": "array", "items": {"type": "string"}},
})

def review_proposal_v2(no):
    resp = client.chat.completions.create(
        model=MODEL, temperature=0,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "esg_review", "strict": True, "schema": REVIEW_SCHEMA}},
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": build_user_message(no)}],
    )
    r = json.loads(resp.choices[0].message.content)
    r["proposal_no"] = no
    return r, resp.usage

OUT_V2 = PROC / "llm_review_v2"
OUT_V2.mkdir(exist_ok=True)
labels = pd.read_csv(PROC / "googl_labels_2026.csv").fillna("").set_index("proposal_no")

results_v2, rows = {}, []
for no in agenda["proposal_no"]:
    try:
        r, usage = review_proposal_v2(no)
        r, v = fix_and_verify(r)
        results_v2[no] = r
        (OUT_V2 / f"googl_p{no:02d}.json").write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")
        draft = f"{labels.loc[no, 'esg_primary']}/{labels.loc[no, 'esg_secondary'] or '없음'}"
        rows.append({"no": no,
                     "esg_llm": f"{r['esg_primary']}/{r['esg_secondary']}",
                     "esg_draft": draft,
                     "primary_match": r["esg_primary"] == labels.loc[no, "esg_primary"],
                     "opinion": r["vote_opinion"]["opinion"],
                     "board_rec": agenda.loc[agenda["proposal_no"] == no, "board_rec"].iloc[0],
                     "evidence": f"{r['evidence_passed']}/{r['evidence_total']}",
                     "needs_check": len(r["needs_check"]),
                     "tokens": usage.total_tokens})
        print(f"{no}번 완료")
    except Exception as e:
        print(f"{no}번 실패: {e}")
    time.sleep(1)

summary_v2 = pd.DataFrame(rows)
print("\n1차 라벨 일치:", summary_v2["primary_match"].sum(), "/", len(summary_v2),
      "/ 근거 통과:", sum(r["evidence_passed"] for r in results_v2.values()),
      "/", sum(r["evidence_total"] for r in results_v2.values()))
summary_v2

1번 완료
2번 완료
3번 완료
4번 완료
5번 완료
6번 완료
7번 완료
8번 완료
9번 완료
10번 완료
11번 완료
12번 완료
13번 완료
14번 완료

1차 라벨 일치: 12 / 14 / 근거 통과: 99 / 113


,no,esg_llm,esg_draft,primary_match,opinion,board_rec,evidence,needs_check,tokens
0,1,G/없음,G/없음,True,찬성,FOR,4/5,1,9519
1,2,G/없음,G/없음,True,찬성,FOR,6/6,0,3053
2,3,G/없음,G/없음,True,찬성,FOR,6/6,0,5952
3,4,G/없음,G/없음,True,찬성,FOR,9/10,1,7911
4,5,E/없음,E/없음,True,찬성,AGAINST,8/9,1,4767
5,6,E/없음,E/없음,True,반대,AGAINST,7/8,1,4678
6,7,G/없음,G/없음,True,찬성,AGAINST,10/10,1,4797
7,8,G/없음,S/G,False,반대,AGAINST,9/9,0,4832
8,9,G/없음,S/없음,False,반대,AGAINST,5/6,1,4652
9,10,S/G,S/없음,True,찬성,AGAINST,5/8,3,4503


## 결과 요약 저장
구조화 출력 실행 결과 요약표를 CSV로 저장한다.

In [21]:
summary_v2.to_csv(OUT_V2 / "summary_v2.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT_V2 / "summary_v2.csv")
print("폴더 파일 수:", len(list(OUT_V2.glob("*"))))

저장: C:\desktop\agent_contest\data\processed\llm_review_v2\summary_v2.csv
폴더 파일 수: 15
